In [ ]:
import os

# 1. 强制清理旧残留
print("正在清理旧文件...")
!rm -rf Diffusion-Illusions
!rm -rf Diffusion-Illusion # 把我之前写错的那个也删掉
!rm -rf master.zip

# 2. 克隆仓库 (注意：这次名字是对的！)
print("正在克隆仓库...")
!git clone https://github.com/RyannDaGreat/Diffusion-Illusions

# 3. 检查是否成功
if os.path.exists('Diffusion-Illusions'):
    print("✅ 仓库克隆成功！")
    
    # 4. 进入目录
    %cd Diffusion-Illusions
    
    # 5. 安装依赖
    print("正在安装依赖 (红色警告请忽略)...")
    !pip install -r requirements.txt
    !pip install mediapy easydict "numpy<2.0"
    
    print("\n✅✅ 环境初始化全部完成！")
    print("⚠️⚠️ 现在的关键步骤：请点击上方菜单 'Runtime' -> 'Restart session' 重启运行时！")
    
else:
    print("❌❌ 克隆还是失败了，请检查网络。")

In [ ]:
import os

# 定义仓库名字（注意带 's'）
repo_name = "Diffusion-Illusions"

# 检查当前是否已经在文件夹里了
if os.getcwd().endswith(repo_name):
    print(f"✅ 当前位置正确: {os.getcwd()}")
else:
    # 如果不在，就尝试进去
    if os.path.exists(repo_name):
        %cd {repo_name}
        print(f"✅ 已切换工作目录到: {os.getcwd()}")
    else:
        # 如果文件夹都不存在，说明之前的克隆没成功，重新克隆一下
        print("⚠️ 文件夹不存在，正在重新克隆...")
        !git clone https://github.com/RyannDaGreat/Diffusion-Illusions
        %cd {repo_name}
        print(f"✅ 克隆并切换完成: {os.getcwd()}")

In [ ]:
from rp import *
import numpy as np
import rp
import torch
import torch.nn as nn
import torch.nn.functional as F
import source.stable_diffusion as sd
from source.learnable_textures import LearnableImageFourier
from source.stable_diffusion_labels import NegativeLabel
from itertools import chain
import torchvision.transforms.functional as TF
import math
from google.colab import files
from PIL import Image, ImageOps

# === 性能优化：预计算与显存常驻 ===

def create_cylindrical_grid_v2(size=256, r_min=0.2, r_max=0.95, device='cuda'):
    """预计算采样网格，避免循环内计算"""
    with torch.no_grad():
        y_range = torch.linspace(-1, 1, size, device=device)
        x_range = torch.linspace(-1, 1, size, device=device)
        grid_y, grid_x = torch.meshgrid(y_range, x_range, indexing='ij')
        
        # 映射到极坐标
        theta = grid_x * math.pi + math.pi / 2 
        r = (1 - (grid_y + 1) / 2) * (r_max - r_min) + r_min
        
        # 转换为采样点 (归一化到 -1 到 1)
        sample_x = r * torch.cos(theta)
        sample_y = r * torch.sin(theta)
        
        # 形状 (1, H, W, 2)
        grid = torch.stack((sample_x, sample_y), dim=2).unsqueeze(0)
    return grid

def create_donut_mask_v2(size=256, r_min=0.2, device='cuda'):
    """使用 Torch 直接在 GPU 上创建遮罩"""
    with torch.no_grad():
        y = torch.linspace(-1, 1, size, device=device)
        x = torch.linspace(-1, 1, size, device=device)
        grid_y, grid_x = torch.meshgrid(y, x, indexing='ij')
        dist = torch.sqrt(grid_x**2 + grid_y**2)
        mask = (dist >= r_min).float().unsqueeze(0)
    return mask

# 高速变换函数
def fast_mirror_reflection(img_tensor, grid):
    # img_tensor 形状应为 (1, C, H, W)
    return F.grid_sample(img_tensor, grid, mode='bilinear', padding_mode='zeros', align_corners=True)

In [ ]:
if 'model_sd' not in dir():
    print("正在加载 Stable Diffusion...")
    model_name = "CompVis/stable-diffusion-v1-4"
    gpu = rp.select_torch_device()
    model_sd = sd.StableDiffusion(gpu, model_name)
    device = model_sd.device
    print("模型加载完毕！")

# 预计算并存入显存
MIRROR_GRID = create_cylindrical_grid_v2(256, r_min=0.2, r_max=0.98, device=device)
DONUT_MASK = create_donut_mask_v2(256, r_min=0.2, device=device)
print("✅ A100 优化网格已就绪")

In [ ]:
print(">>> 请点击下方按钮上传你的'谜底'图片 (例如《你的名字》人物合影) <<<")
print("提示：最好是正方形构图，或者主体在中间的图片。")
uploaded = files.upload()

if uploaded:
    filename = next(iter(uploaded))
    
    # 1. 读取并缩放到 256x256
    target_pil = Image.open(filename).convert('RGB')
    target_pil = ImageOps.fit(target_pil, (256, 256), method=Image.Resampling.LANCZOS)
    
    # 2. 转为 Tensor
    target_tensor = TF.to_tensor(target_pil).to(device)
    
    print("\n目标图片处理完毕！这将会是在圆柱镜中看到的画面：")
    rp.display_image(rp.as_numpy_image(target_tensor))
else:
    print("❌ 未上传图片，请重新运行此块！")

In [ ]:
# === 🎮 游戏参数 ===
GUIDANCE_STRENGTH = 3000 # 隐写强度：越大镜中像越清晰，但地面图越难看

# === 🎨 画面描述 ===
# 地面上的图：看似乱码，我们引导它生成“星空、彗星轨迹、抽象线条”
# 这样极向拉伸的纹理看起来就像流星雨
prompt_canvas = "Abstract shooting stars, comet trails in night sky, beautiful blue and pink streaks, meteor shower, long exposure photography, 8k texture"

negative_prompt = "blur, low quality, ugly, distortion, text, watermark, realistic faces"

# === 初始化可训练图像 ===
# 只生成一张图 (画布)
image_maker = lambda: LearnableImageFourier(height=256, width=256, hidden_dim=256, num_features=256).to(device)

raw_canvas = image_maker()

# 定义获取图像的函数 (应用甜甜圈遮罩，中间挖空放杯子)
get_canvas = lambda: raw_canvas() * DONUT_MASK

# 准备标签
label_canvas = NegativeLabel(prompt_canvas, negative_prompt)

# 优化器
optim = torch.optim.SGD(raw_canvas.parameters(), lr=1e-4)

print(f"初始化完成。准备将图像隐藏在彗星轨迹中...")

In [ ]:
# 提前将目标图处理为标准 4D Tensor (1, 3, 256, 256)
target_tensor_4d = target_tensor.unsqueeze(0).to(device)

NUM_ITER = 3000           
DISPLAY_INTERVAL = 200    

model_sd.max_step = 980
model_sd.min_step = 20
display_eta = rp.eta(NUM_ITER, title='A100 High-Speed Training')

print(f"🚀 开始高速训练...")

try:
    for iter_num in range(NUM_ITER):
        # --- A. 正常的 SD 训练 ---
        curr_canvas = get_canvas()
        # SD 的 train_step 内部通常已经优化，确保输入是 (1, C, H, W)
        _ = model_sd.train_step(
            label_canvas.embedding,
            curr_canvas[None], # 增加 batch 维度
            noise_coef=0.1,
            guidance_scale=50
        )

        # --- B. 极速 Anamorphosis Loss ---
        # 1. 模拟反射 (纯 GPU 操作，极快)
        # 注意：curr_canvas[None] 形状为 (1, 3, 256, 256)
        simulated_reflection = fast_mirror_reflection(curr_canvas[None], MIRROR_GRID)
        
        # 2. 直接计算 MSE (全部在显存内)
        loss_illusion = F.mse_loss(simulated_reflection, target_tensor_4d) * GUIDANCE_STRENGTH
        
        # 3. 反向传播
        loss_illusion.backward()

        # --- C. 异步显示 (避免拖慢速度) ---
        if iter_num % DISPLAY_INTERVAL == 0:
            with torch.no_grad():
                # 只有显示的时候才把数据往 CPU 传一次
                canvas_np = rp.as_numpy_image(curr_canvas)
                reflection_np = rp.as_numpy_image(simulated_reflection.squeeze(0))
                
                from IPython.display import clear_output
                clear_output(wait=True)
                print(f"Iteration {iter_num} | 正在高速运行...")
                rp.display_image(np.hstack([canvas_np, reflection_np]))

        optim.step()
        optim.zero_grad(set_to_none=True) # A100 推荐用法，更省显存

except KeyboardInterrupt:
    print("已停止。")

In [ ]:
print("==== 最终成果展示 ====")

final_canvas = get_canvas()
final_reflection = apply_mirror_reflection(final_canvas, MIRROR_GRID)

print("1. 请把这张图打印出来 (或者显示在iPad上平放桌面上):")
rp.display_image(rp.as_numpy_image(final_canvas))

print("\n2. 把一个反光圆柱体(不锈钢杯子/镜面纸卷成的筒)放在图片正中心。")
print("   你在杯壁上看到的画面应该是这样的：")
rp.display_image(rp.as_numpy_image(final_reflection))

print("\n3. 保存结果：")
# 保存图片
img_pil = TF.to_pil_image(final_canvas.cpu().clamp(0,1))
img_pil.save("anamorphosis_result.png")
print("✅ 已保存为 anamorphosis_result.png")